# EX_07 — Reranking y optimización (ejercicios)

**Notebook de referencia:** `notebook/07_Reranking_Optimizacion.ipynb`

**Tiempo orientativo:** ~30 minutos.


## Actividad 1 — Reordenar por cross-score simulado

Dada una query y 5 documentos, supón que tienes scores de un bi-encoder (baratos) y scores de un cross-encoder (caros). Implementa: tomar top-4 por bi-encoder y reordenar solo esos 4 por cross-score.


In [1]:
import numpy as np

query = "latency vs throughput"
docs = [
    "doc0: latency is the delay of a single request",
    "doc1: throughput measures total data processed per second",
    "doc2: how to cook pasta quickly",
    "doc3: network bandwidth and optimization guide",
    "doc4: storage systems and basic performance metrics"
]

# Scores iniciales del Bi-Encoder y del Cross-Encoder para cada documento
bi_scores = np.array([0.72, 0.81, 0.55, 0.78, 0.60])
cross_scores = np.array([0.1, 0.9, 0.2, 0.85, 0.3])

# --- ETAPA 1: Filtrado inicial con Bi-Encoder (Top-4) ---
# Obtenemos los índices ordenados de mayor a menor según el Bi-Encoder
bi_sorted_indices = np.argsort(bi_scores)[::-1]
# Nos quedamos únicamente con los mejores 4 índices
top_4_indices = bi_sorted_indices[:4]

print("--- Etapa 1: Top-4 seleccionados por Bi-Encoder ---")
for idx in top_4_indices:
    print(f"Índice {idx} | {docs[idx]} (Bi-Score: {bi_scores[idx]})")


# --- ETAPA 2: Reordenación (Reranking) con Cross-Encoder ---
# Extraemos los scores del Cross-Encoder correspondientes solo a esos 4 seleccionados
selected_cross_scores = cross_scores[top_4_indices]

# Obtenemos el orden interno relativo de estos 4 elementos según el Cross-Encoder
rerank_relative_indices = np.argsort(selected_cross_scores)[::-1]

# Mapeamos los índices relativos de vuelta a los índices originales del repositorio de documentos
final_indices = top_4_indices[rerank_relative_indices]


# --- RESULTADO FINAL ---
print("\n--- Etapa 2: Ranking Final tras el Reranking ---")
for rank, idx in enumerate(final_indices, 1):
    print(f"Puesto {rank}: Índice {idx} | {docs[idx]} (Cross-Score: {cross_scores[idx]})")


--- Etapa 1: Top-4 seleccionados por Bi-Encoder ---
Índice 1 | doc1: throughput measures total data processed per second (Bi-Score: 0.81)
Índice 3 | doc3: network bandwidth and optimization guide (Bi-Score: 0.78)
Índice 0 | doc0: latency is the delay of a single request (Bi-Score: 0.72)
Índice 4 | doc4: storage systems and basic performance metrics (Bi-Score: 0.6)

--- Etapa 2: Ranking Final tras el Reranking ---
Puesto 1: Índice 1 | doc1: throughput measures total data processed per second (Cross-Score: 0.9)
Puesto 2: Índice 3 | doc3: network bandwidth and optimization guide (Cross-Score: 0.85)
Puesto 3: Índice 4 | doc4: storage systems and basic performance metrics (Cross-Score: 0.3)
Puesto 4: Índice 0 | doc0: latency is the delay of a single request (Cross-Score: 0.1)


## Actividad 2 — MMR esquemático

En pseudocódigo en Python (sin librería), bosqueja 5 líneas de selección **MMR** (balance relevancia / diversidad).


In [3]:
# TODO: MMR pseudocode as comments or stub function
## Actividad 2 — MMR esquemático
def mmr_selection(query_emb, candidate_embs, k=3, lmbda=0.5):
    selected = [candidate_embs.pop(0)]  # 1. Empezamos con el más relevante
    
    while len(selected) < k:
        # 2. Calculamos relevancia vs query para todos los restantes
        rel = [cosine_sim(c, query_emb) for c in candidate_embs]
        
        # 3. Calculamos redundancia máxima con lo ya seleccionado
        dist = [max([cosine_sim(c, s) for s in selected]) for c in candidate_embs]
        
        # 4. Ecuación MMR: Balance entre novedad y utilidad
        mmr_scores = [lmbda * r - (1 - lmbda) * d for r, d in zip(rel, dist)]
        
        # 5. Extraemos el mejor y repetimos
        selected.append(candidate_embs.pop(np.argmax(mmr_scores)))
        
    return selected

## Actividad 3 — Latencia

Estima en markdown (tabla breve) coste relativo: embedding único de query, k llamadas cross-encoder, generación LLM 200 tokens.


| Etapa | Coste relativo (tú eliges escala) |
|--------|-------------------------------------|
| EMBEDDING DE QUERY | 1x |
| K LLAMADAS | 10x - 50X |
| GENERACIÓN LLM | 200x - 1000x |
